In [ ]:
# 1. Generador (Stream): Simular eventos continuos de red (login, view, logout) desde distintas IPs.
# 2. Procesador (WAZ): Analizar el stream de eventos en tiempo real.
# 3. Regla de Detección: Identificar como anomalía si una misma IP realiza múltiples intentos de 'login'
#    (ej. >3 intentos) en una ventana de tiempo corta (ej. 10 segundos).
# 4. Acción de Mitigación: Añadir la IP infractora a una lista de bloqueo temporal (ej. durante 5 segundos)
#    y descartar todos sus eventos futuros hasta que expire dicho bloqueo.

import time
import random
from datetime import datetime
from collections import deque

# Códigos de color ANSI para la terminal
COLOR_VERDE = "\033[92m"
COLOR_NARANJA = "\033[38;5;208m"
COLOR_AMARILLO = "\033[33m"
COLOR_ROJO = "\033[91m"
COLOR_AZUL = "\033[94m"
COLOR_RESET = "\033[0m"

class SimuladorTrafico:
    """Genera un flujo constante de eventos simulados en la red."""
    def __init__(self):
        # Limitamos a 3 IPs para forzar que coincidan y salten las alertas
        self.ips = [f"192.168.1.{i}" for i in range(1, 4)]
        self.acciones = ['login', 'view', 'logout']

    def generar_evento(self):
        """Crea un evento individual con una acción aleatoria."""
        return {
            'hora_legible': datetime.now().strftime("%H:%M:%S"),
            'tiempo_exacto': time.time(), # Usado para cálculos matemáticos
            'ip': random.choice(self.ips),
            # Damos un 60% de probabilidad al 'login' para asegurar que ocurran ataques
            'accion': random.choices(self.acciones, weights=[0.6, 0.3, 0.1])[0]
        }

    def iniciar_flujo(self, total_eventos=75, retraso_maximo=0.9):
        """Generador que emite eventos con una pequeña pausa entre ellos."""
        for _ in range(total_eventos):
            yield self.generar_evento()
            time.sleep(random.uniform(0.1, retraso_maximo))

class AnalizadorWaz:
    """Procesa los eventos en tiempo real y aplica reglas de bloqueo."""
    def __init__(self, limite_logins=4, ventana_segundos=10, tiempo_bloqueo=6):
        self.limite_logins = limite_logins
        self.ventana_segundos = ventana_segundos
        self.tiempo_bloqueo = tiempo_bloqueo

        # Diccionarios para mantener el estado de cada IP en la memoria
        self.historial_logins = {}
        self.ips_bloqueadas = {}

    def evaluar(self, evento):
        """Recibe un evento y decide si lo permite, lo bloquea o emite una alerta."""
        ip = evento['ip']
        accion = evento['accion']
        tiempo_actual = evento['tiempo_exacto']
        mensajes = []

        # --- PASO 1: Control de bloqueos activos ---
        if ip in self.ips_bloqueadas:
            # Si el tiempo actual superó el tiempo de castigo, lo perdonamos
            if tiempo_actual > self.ips_bloqueadas[ip]:
                del self.ips_bloqueadas[ip]
                mensajes.append(f"{COLOR_AZUL}DESBLOQUEO: Tiempo expirado. IP {ip} vuelve a tener acceso.{COLOR_RESET}")
            else:
                # Si sigue bloqueado, calculamos cuánto le queda y descartamos su evento
                tiempo_restante = round(self.ips_bloqueadas[ip] - tiempo_actual, 1)
                return [f"{COLOR_ROJO}BLOQUEADO ({tiempo_restante}s restantes): Evento '{accion}' de IP {ip} descartado.{COLOR_RESET}"]

        # --- PASO 2: Análisis de intentos de Login ---
        if accion == 'login':
            # Si es la primera vez que esta IP hace login, le creamos su historial
            if ip not in self.historial_logins:
                self.historial_logins[ip] = deque()

            # Limpiamos el historial: borramos los logins que sean más antiguos que nuestra ventana (10s)
            while self.historial_logins[ip] and (tiempo_actual - self.historial_logins[ip][0] > self.ventana_segundos):
                self.historial_logins[ip].popleft()

            # Registramos el nuevo intento de login
            self.historial_logins[ip].append(tiempo_actual)
            intentos_actuales = len(self.historial_logins[ip])

            # --- PASO 3: Aplicación de la regla (Castigo) ---
            if intentos_actuales >= self.limite_logins:
                # Se pasó del límite: calculamos hasta cuándo estará bloqueado y limpiamos su historial
                self.ips_bloqueadas[ip] = tiempo_actual + self.tiempo_bloqueo
                self.historial_logins[ip].clear()
                mensajes.append(f"{COLOR_NARANJA}ALERTA WAZ: >= {self.limite_logins} logins en {self.ventana_segundos}s. IP {ip} bloqueada por {self.tiempo_bloqueo}s.{COLOR_RESET}")
            else:
                # Aún no llega al límite, pero avisamos del intento
                mensajes.append(f"{COLOR_AMARILLO}ATENCIÓN: Intento de login ({intentos_actuales}/{self.limite_logins} en margen de {self.ventana_segundos}s) desde {ip}{COLOR_RESET}")

        # --- PASO 4: Otras acciones (view, logout) ---
        else:
            mensajes.append(f"{COLOR_VERDE}NORMAL: {accion} desde {ip}{COLOR_RESET}")

        return mensajes

# ==========================================
# EJECUCIÓN PRINCIPAL DEL SCRIPT
# ==========================================
if __name__ == "__main__":
    simulador = SimuladorTrafico()

    # Configuración del WAZ: Máximo 4 logins en 10 segundos. Castigo: 6 segundos.
    waz = AnalizadorWaz(limite_logins=4, ventana_segundos=10, tiempo_bloqueo=6)

    # Procesamos el flujo en vivo
    for evento in simulador.iniciar_flujo(total_eventos=75, retraso_maximo=0.9):
        resultados = waz.evaluar(evento)

        for mensaje in resultados:
            print(f"[{evento['hora_legible']}] {mensaje}")

[12:44:22] ATENCIÓN: Intento de login (1/4 en margen de 10s) desde 192.168.1.2
[12:44:23] NORMAL: logout desde 192.168.1.2
[12:44:23] ATENCIÓN: Intento de login (2/4 en margen de 10s) desde 192.168.1.2
[12:44:24] ATENCIÓN: Intento de login (1/4 en margen de 10s) desde 192.168.1.3
[12:44:24] ATENCIÓN: Intento de login (3/4 en margen de 10s) desde 192.168.1.2
[12:44:24] ALERTA WAZ: >= 4 logins en 10s. IP 192.168.1.2 bloqueada por 6s.
[12:44:25] ATENCIÓN: Intento de login (2/4 en margen de 10s) desde 192.168.1.3
[12:44:25] ATENCIÓN: Intento de login (1/4 en margen de 10s) desde 192.168.1.1
[12:44:26] BLOQUEADO (4.3s restantes): Evento 'login' de IP 192.168.1.2 descartado.
[12:44:26] ATENCIÓN: Intento de login (3/4 en margen de 10s) desde 192.168.1.3
[12:44:27] ALERTA WAZ: >= 4 logins en 10s. IP 192.168.1.3 bloqueada por 6s.
[12:44:27] NORMAL: view desde 192.168.1.1
[12:44:28] BLOQUEADO (2.5s restantes): Evento 'view' de IP 192.168.1.2 descartado.
[12:44:29] ATENCIÓN: Intento de login (2/4